# Video Telemetry QoS Analytics Pipeline - Production Monitoring

## Purpose
Real-time monitoring and health checks for the production streaming pipeline.

## Monitoring Coverage
1. **Pipeline Health** - Latest runs, failures, performance
2. **Data Freshness** - Lag detection across bronze/silver/gold layers
3. **Data Quality** - Quarantine trends and validation failures
4. **QoS Metrics** - Business KPI health and anomaly detection
5. **Resource Utilization** - Storage growth and compute efficiency

## Alert Thresholds
* ⚠️ **Critical**: Pipeline failed, data lag >2 hours, quarantine rate >10%
* ⚡ **Warning**: Data lag >1 hour, quarantine rate >5%, QoS score degradation
* ℹ️ **Info**: Normal operations

## Usage
Run all cells to generate current status. Schedule as a job for continuous monitoring.

In [0]:
# Monitoring configuration
from datetime import datetime, timedelta
from pyspark.sql import functions as F

# Pipeline configuration
PIPELINE_ID = "4296766b-887a-4ab4-9eef-2e6e07a32b66"
PIPELINE_NAME = "Video Telemetry QoS Analytics Pipeline"
CATALOG = "workspace"
SCHEMA = "hive_video_analytics"

# Alert thresholds
THRESHOLDS = {
    "data_lag_warning_minutes": 60,
    "data_lag_critical_minutes": 120,
    "quarantine_rate_warning_pct": 5.0,
    "quarantine_rate_critical_pct": 10.0,
    "qos_poor_threshold": 40.0,
    "qos_warning_pct_customers": 10.0,
    "storage_growth_warning_gb_per_day": 100
}

# Tables to monitor
TABLES = {
    "bronze": f"{CATALOG}.{SCHEMA}.bronze_telemetry_raw",
    "silver": f"{CATALOG}.{SCHEMA}.silver_telemetry_enriched",
    "quarantine": f"{CATALOG}.{SCHEMA}.silver_telemetry_quarantine",
    "gold": f"{CATALOG}.{SCHEMA}.gold_viewer_qos_metrics_streaming"
}

print(f"✓ Monitoring configuration loaded")
print(f"  Pipeline: {PIPELINE_NAME}")
print(f"  Tables: {len(TABLES)} tables")
print(f"  Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
%sql
-- Check latest pipeline runs and identify failures
SELECT 
  update_id,
  state,
  creation_time,
  CASE 
    WHEN state = 'COMPLETED' THEN '✓'
    WHEN state = 'FAILED' THEN '✗'
    WHEN state = 'RUNNING' THEN '⟳'
    ELSE '⚠'
  END as status_icon,
  TIMESTAMPDIFF(MINUTE, creation_time, COALESCE(completion_time, CURRENT_TIMESTAMP())) as duration_minutes
FROM system.lakeflow.pipeline_runs
WHERE pipeline_id = '4296766b-887a-4ab4-9eef-2e6e07a32b66'
ORDER BY creation_time DESC
LIMIT 10;

In [0]:
%sql
-- Check data freshness in bronze layer
-- Alert if latest data is older than threshold
SELECT 
  'bronze_telemetry_raw' as layer,
  MAX(timestamp) as latest_event_timestamp,
  FROM_UNIXTIME(MAX(timestamp_server) / 1000) as latest_server_time,
  TIMESTAMPDIFF(MINUTE, FROM_UNIXTIME(MAX(timestamp_server) / 1000), CURRENT_TIMESTAMP()) as lag_minutes,
  CASE 
    WHEN TIMESTAMPDIFF(MINUTE, FROM_UNIXTIME(MAX(timestamp_server) / 1000), CURRENT_TIMESTAMP()) < 60 THEN '✓ Fresh'
    WHEN TIMESTAMPDIFF(MINUTE, FROM_UNIXTIME(MAX(timestamp_server) / 1000), CURRENT_TIMESTAMP()) < 120 THEN '⚠ Warning'
    ELSE '✗ Critical Lag'
  END as status,
  COUNT(*) as total_records,
  COUNT(DISTINCT eventDate) as date_partitions
FROM workspace.hive_video_analytics.bronze_telemetry_raw
WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS;

In [0]:
%sql
-- Check silver layer freshness and compare to bronze
SELECT 
  'silver_telemetry_enriched' as layer,
  MAX(timestamp) as latest_event_timestamp,
  FROM_UNIXTIME(MAX(timestamp_server) / 1000) as latest_server_time,
  TIMESTAMPDIFF(MINUTE, FROM_UNIXTIME(MAX(timestamp_server) / 1000), CURRENT_TIMESTAMP()) as lag_minutes,
  CASE 
    WHEN TIMESTAMPDIFF(MINUTE, FROM_UNIXTIME(MAX(timestamp_server) / 1000), CURRENT_TIMESTAMP()) < 60 THEN '✓ Fresh'
    WHEN TIMESTAMPDIFF(MINUTE, FROM_UNIXTIME(MAX(timestamp_server) / 1000), CURRENT_TIMESTAMP()) < 120 THEN '⚠ Warning'
    ELSE '✗ Critical Lag'
  END as status,
  COUNT(*) as total_records,
  COUNT(DISTINCT customerId) as unique_customers
FROM workspace.hive_video_analytics.silver_telemetry_enriched
WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS;

In [0]:
%sql
-- Check gold layer aggregations freshness
SELECT 
  'gold_viewer_qos_metrics' as layer,
  MAX(eventDate) as latest_date,
  DATEDIFF(CURRENT_DATE(), MAX(eventDate)) as days_behind,
  CASE 
    WHEN MAX(eventDate) = CURRENT_DATE() THEN '✓ Current'
    WHEN MAX(eventDate) = CURRENT_DATE() - INTERVAL 1 DAY THEN '⚠ 1 Day Behind'
    ELSE '✗ Stale'
  END as status,
  COUNT(DISTINCT customerId) as unique_customers,
  COUNT(DISTINCT clientId) as unique_clients,
  SUM(total_sessions) as total_sessions_aggregated
FROM workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming
WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS;

In [0]:
%sql
-- Monitor data quality issues over time
-- Alert if quarantine rate exceeds thresholds
WITH daily_metrics AS (
  SELECT 
    eventDate,
    COUNT(*) as quarantine_count,
    COUNT(DISTINCT customerId) as affected_customers
  FROM workspace.hive_video_analytics.silver_telemetry_quarantine
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
  GROUP BY eventDate
),
daily_totals AS (
  SELECT 
    eventDate,
    COUNT(*) as total_records
  FROM workspace.hive_video_analytics.bronze_telemetry_raw
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
  GROUP BY eventDate
)
SELECT 
  m.eventDate,
  m.quarantine_count,
  t.total_records,
  ROUND((m.quarantine_count * 100.0) / NULLIF(t.total_records, 0), 2) as quarantine_rate_pct,
  m.affected_customers,
  CASE 
    WHEN (m.quarantine_count * 100.0) / NULLIF(t.total_records, 0) < 5 THEN '✓ Normal'
    WHEN (m.quarantine_count * 100.0) / NULLIF(t.total_records, 0) < 10 THEN '⚠ Warning'
    ELSE '✗ Critical'
  END as status
FROM daily_metrics m
JOIN daily_totals t ON m.eventDate = t.eventDate
ORDER BY m.eventDate DESC;

In [0]:
%sql
-- Identify most common data quality issues
SELECT 
  quarantine_reason,
  COUNT(*) as failure_count,
  COUNT(DISTINCT customerId) as affected_customers,
  ROUND((COUNT(*) * 100.0) / SUM(COUNT(*)) OVER (), 2) as pct_of_total,
  MIN(eventDate) as first_seen,
  MAX(eventDate) as last_seen
FROM workspace.hive_video_analytics.silver_telemetry_quarantine
WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY quarantine_reason
ORDER BY failure_count DESC;

In [0]:
%sql
-- Monitor QoS score distribution across customer base
SELECT 
  qos_category,
  COUNT(DISTINCT customerId) as customer_count,
  ROUND((COUNT(DISTINCT customerId) * 100.0) / SUM(COUNT(DISTINCT customerId)) OVER (), 2) as pct_customers,
  ROUND(AVG(qos_score), 2) as avg_score,
  ROUND(AVG(total_buffering_time_sec), 2) as avg_buffering_sec,
  ROUND(AVG(total_data_consumed_mb), 2) as avg_data_mb,
  CASE 
    WHEN qos_category = 'Poor' AND (COUNT(DISTINCT customerId) * 100.0) / SUM(COUNT(DISTINCT customerId)) OVER () > 10 THEN '✗ High Poor QoS'
    WHEN qos_category = 'Poor' AND (COUNT(DISTINCT customerId) * 100.0) / SUM(COUNT(DISTINCT customerId)) OVER () > 5 THEN '⚠ Elevated Poor QoS'
    ELSE '✓ OK'
  END as alert_status
FROM workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming
WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY qos_category
ORDER BY 
  CASE qos_category
    WHEN 'Excellent' THEN 1
    WHEN 'Good' THEN 2
    WHEN 'Fair' THEN 3
    WHEN 'Poor' THEN 4
  END;

In [0]:
%sql
-- Identify customers with declining QoS trends
WITH daily_qos AS (
  SELECT 
    customerId,
    eventDate,
    AVG(qos_score) as avg_qos_score,
    SUM(total_buffering_time_sec) as total_buffering_sec
  FROM workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming
  WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
  GROUP BY customerId, eventDate
),
trend_analysis AS (
  SELECT 
    customerId,
    AVG(CASE WHEN eventDate >= CURRENT_DATE() - INTERVAL 3 DAYS THEN avg_qos_score END) as recent_qos,
    AVG(CASE WHEN eventDate < CURRENT_DATE() - INTERVAL 3 DAYS THEN avg_qos_score END) as historical_qos,
    SUM(CASE WHEN eventDate >= CURRENT_DATE() - INTERVAL 3 DAYS THEN total_buffering_sec END) as recent_buffering,
    SUM(CASE WHEN eventDate < CURRENT_DATE() - INTERVAL 3 DAYS THEN total_buffering_sec END) as historical_buffering
  FROM daily_qos
  GROUP BY customerId
  HAVING recent_qos IS NOT NULL AND historical_qos IS NOT NULL
)
SELECT 
  customerId,
  ROUND(historical_qos, 2) as baseline_qos_score,
  ROUND(recent_qos, 2) as recent_qos_score,
  ROUND(recent_qos - historical_qos, 2) as qos_change,
  ROUND(((recent_qos - historical_qos) / NULLIF(historical_qos, 0)) * 100, 2) as qos_change_pct,
  ROUND(recent_buffering, 2) as recent_buffering_sec,
  CASE 
    WHEN (recent_qos - historical_qos) < -20 THEN '✗ Severe Degradation'
    WHEN (recent_qos - historical_qos) < -10 THEN '⚠ Moderate Degradation'
    WHEN (recent_qos - historical_qos) > 10 THEN '✓ Improvement'
    ELSE '→ Stable'
  END as trend_status
FROM trend_analysis
WHERE ABS(recent_qos - historical_qos) > 5  -- Only show significant changes
ORDER BY qos_change ASC
LIMIT 100;

In [0]:
%sql
-- Monitor storage growth and file counts
SELECT 
  name as table_name,
  ROUND(sizeInBytes / (1024.0 * 1024.0 * 1024.0), 2) as size_gb,
  numFiles as file_count,
  ROUND(sizeInBytes / NULLIF(numFiles, 0) / (1024.0 * 1024.0), 2) as avg_file_size_mb,
  properties['delta.lastCommitTimestamp'] as last_update,
  CASE 
    WHEN numFiles > 10000 THEN '⚠ High File Count - Run OPTIMIZE'
    WHEN ROUND(sizeInBytes / NULLIF(numFiles, 0) / (1024.0 * 1024.0), 2) < 10 THEN '⚠ Small Files - Run OPTIMIZE'
    ELSE '✓ OK'
  END as optimization_status
FROM (
  SELECT * FROM (
    DESCRIBE DETAIL workspace.hive_video_analytics.bronze_telemetry_raw
    UNION ALL
    DESCRIBE DETAIL workspace.hive_video_analytics.silver_telemetry_enriched
    UNION ALL
    DESCRIBE DETAIL workspace.hive_video_analytics.silver_telemetry_quarantine
    UNION ALL
    DESCRIBE DETAIL workspace.hive_video_analytics.gold_viewer_qos_metrics_streaming
  )
)
ORDER BY size_gb DESC;

In [0]:
%sql
-- Track daily record ingestion rates
SELECT 
  eventDate,
  COUNT(*) as records_ingested,
  COUNT(DISTINCT customerId) as unique_customers,
  COUNT(DISTINCT contentId) as unique_content,
  ROUND(SUM(player.bufferingTime) / 1000.0, 2) as total_buffering_sec,
  ROUND((SUM(totalDistribution.sourceTraffic.receivedData) + SUM(totalDistribution.p2pTraffic.receivedData)) / (1024.0 * 1024.0 * 1024.0), 2) as total_data_gb
FROM workspace.hive_video_analytics.bronze_telemetry_raw
WHERE eventDate >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY eventDate
ORDER BY eventDate DESC;

# Monitoring Summary

## Key Health Indicators
Review the cells above for:

* ✓ **Pipeline Status**: All recent runs successful
* ✓ **Data Freshness**: All layers current (lag < 1 hour)
* ✓ **Quarantine Rate**: < 5% (normal operations)
* ✓ **QoS Health**: < 10% customers in Poor category
* ✓ **Storage**: Optimized file sizes, manageable growth

## Alert Response Procedures

### Critical Alerts (✗)

**Pipeline Failed**
1. Check system.lakeflow.pipeline_event_log for error details
2. Review quarantine table for data quality spike
3. Verify source data availability in /Volumes/workspace/default/hivestreamdata/
4. Contact data engineering on-call

**Data Lag > 2 Hours**
1. Check if pipeline is running (may be stopped)
2. Verify Auto Loader checkpoint health
3. Check for source data delivery delays
4. Restart pipeline if needed

**Quarantine Rate > 10%**
1. Query quarantine breakdown by reason
2. Identify affected customers/content
3. Contact upstream data provider
4. May need to halt pipeline if data corruption is widespread

### Warning Alerts (⚠)

**Data Lag 1-2 Hours**
1. Monitor for auto-recovery
2. Check compute resource availability
3. Alert if persists > 30 minutes

**Quarantine Rate 5-10%**
1. Review quarantine trends
2. Identify if specific to certain customers/dates
3. Document for upstream team

**QoS Degradation**
1. Check for CDN/P2P infrastructure issues
2. Correlate with content or customer segments
3. Notify streaming operations team

## Scheduling This Notebook

Schedule to run every 30 minutes during business hours:

```bash
databricks jobs create --json '{
  "name": "Pipeline Monitoring - Every 30min",
  "tasks": [{
    "task_key": "monitoring",
    "notebook_task": {
      "notebook_path": "/Users/vasuambi@gmail.com/video_telemetry_qos_analytics_pipeline_1ff1d722/operations/Pipeline_Monitoring_Dashboard"
    },
    "new_cluster": {
      "spark_version": "13.3.x-scala2.12",
      "node_type_id": "i3.xlarge",
      "num_workers": 0
    }
  }],
  "schedule": {
    "quartz_cron_expression": "0 0/30 * * * ?",
    "timezone_id": "UTC"
  },
  "email_notifications": {
    "on_failure": ["data-eng-team@company.com"]
  }
}'
```

## Dashboard Integration

Consider creating a Lakeview dashboard from these queries for real-time visualization.